# LangGraph Streaming Project (Groq Llama-3 Version)

This notebook demonstrates how to implement a stateful multi-node agent workflow using **LangGraph** and stream both **graph state updates** and **real-time LLM tokens** using the high-speed **Groq API** (`llama-3.1-8b-instant` model).

### What We Will Build:
We will build a **Research Assistant** consisting of a 2-node graph:
1. **Researcher Node**: Analyzes a complex topic, breaks it down, and writes a detailed outline of key areas.
2. **Writer Node**: Takes the researcher's outline and writes a polished, structured article on the topic.

### Streaming Modes Demonstrated:
1. **Node Transition Streaming**: See which node is active and check the updated state as the execution flows.
2. **Token-by-Token Streaming**: Stream the LLM's text output in real-time as it generates tokens.

In [1]:
# Install the required packages
!pip install langgraph groq python-dotenv -q --no-warn-script-location

In [2]:
import os
import time
from dotenv import load_dotenv
from groq import Groq

# Load environment variables from .env
# .env is located in the parent directory of agent_workflows/
load_dotenv(dotenv_path="../.env")

groq_key = os.getenv("GROQ_API_KEY")
if not groq_key:
    print("❌ ERROR: GROQ_API_KEY is not set. Please check your .env file.")
else:
    print("✅ GROQ_API_KEY loaded successfully. Initializing Groq client...")
    client = Groq()

✅ GROQ_API_KEY loaded successfully. Initializing Groq client...


## Define Graph State and Custom Nodes

We define the shared state using a `TypedDict` and the nodes that represent our agent's steps.

In [3]:
from typing import TypedDict, List
from langgraph.graph import StateGraph, START, END

# 1. Define the Shared State
class AgentState(TypedDict):
    topic: str
    research_notes: str
    final_article: str

# Helper function to call Groq with token streaming
def call_groq_streaming(system_prompt: str, user_prompt: str, node_name: str):
    print(f"\n--- [Streaming from Groq inside {node_name}] ---")
    stream = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        temperature=0.7,
        stream=True
    )
    
    full_response = []
    for chunk in stream:
        content = chunk.choices[0].delta.content
        if content:
            # Print token in real-time to stdout
            print(content, end="", flush=True)
            full_response.append(content)
            
    print("\n-----------------------------------------------\n")
    return "".join(full_response)

# 2. Define the Researcher Node
def researcher_node(state: AgentState):
    print(f"🕵️ Researcher Node starting for topic: '{state['topic']}'")
    system_prompt = "You are an expert researcher. Provide a bulleted outline of key areas, historical context, and recent trends for the given topic."
    user_prompt = f"Research Topic: {state['topic']}"
    
    research_notes = call_groq_streaming(system_prompt, user_prompt, "Researcher Node")
    return {"research_notes": research_notes}

# 3. Define the Writer Node
def writer_node(state: AgentState):
    print("✍️ Writer Node starting using research notes...")
    system_prompt = "You are a professional technical writer. Write a comprehensive, clear, and engaging article based on the research notes provided. Use proper Markdown headings."
    user_prompt = f"Research Notes:\n{state['research_notes']}"
    
    final_article = call_groq_streaming(system_prompt, user_prompt, "Writer Node")
    return {"final_article": final_article}

## Assemble the State Graph

Now, we assemble our nodes into a compiled graph.

In [4]:
# 1. Initialize StateGraph
builder = StateGraph(AgentState)

# 2. Add Nodes
builder.add_node("researcher", researcher_node)
builder.add_node("writer", writer_node)

# 3. Add Edges
builder.add_edge(START, "researcher")
builder.add_edge("researcher", "writer")
builder.add_edge("writer", END)

# 4. Compile the Graph
graph = builder.compile()
print("✅ LangGraph compiled successfully!")

✅ LangGraph compiled successfully!


## Test: Real-Time Token Streaming + Node Outputs

Here, we run the graph and observe:
1. **Real-time token streaming** as each node talks to Groq.
2. **Graph state updates** printed after each node finishes its work.

In [5]:
initial_state = {"topic": "The Impact of Quantum Computing on Cryptography"}

print("🚀 Executing LangGraph with streaming...\n")

# Use graph.stream to capture node updates
for output in graph.stream(initial_state, stream_mode="updates"):
    for node_name, state_update in output.items():
        print(f"📌 [Node Complete] Name: '{node_name}' finished.")
        # If we updated state variables, print them
        for key, val in state_update.items():
            print(f"   -> Updated state key '{key}' (Length of text: {len(val)} characters)")
        print("-" * 50)

🚀 Executing LangGraph with streaming...

🕵️ Researcher Node starting for topic: 'The Impact of Quantum Computing on Cryptography'

--- [Streaming from Groq inside Researcher Node] ---
**

Topic

:**

 The

 Impact

...

 quantum

 computing

 advancements

.


-----------------------------------------------

📌 [Node Complete] Name: 'writer' finished.
   -> Updated state key 'final_article' (Length of text: 6162 characters)
--------------------------------------------------
